## Query Tracker Cross-Language Test

Validates query ID capture and RESULT_SCAN reuse across Python, SQL, R, and Scala.

The query tracker hooks into `ServerConnection.add_query_listener()` to
implicitly capture every query ID. This notebook proves that a query executed
in one language can be consumed in another via `RESULT_SCAN()`.


## 0. Setup

Install R, Scala, snowflakeR, bootstrap Scala, then install the query tracker.
Tracker must be installed AFTER all language runtimes to avoid callback
interference during JVM bootstrap.


In [ ]:
from sfnb_setup import setup_notebook
result = setup_notebook(config="query_tracker_test_config.yaml")

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

from scala_helpers import setup_scala_environment, bootstrap_snowpark_scala
setup_scala_environment()
bootstrap_snowpark_scala(session)
print(f"Session ready: {session.get_current_warehouse()}")

In [ ]:
from query_tracker import install_query_tracker, nb_last_query_id, nb_query_id, get_registry
tracker = install_query_tracker(session)
print(f"Query tracker installed: {tracker is not None}")

## 1. SQL Cell Capture

Native SQL cell.  The tracker's `_notify` callback captures the query ID
into the cell buffer.  When the next Python cell starts, `_pre_run_cell`
flushes orphaned buffer entries so `nb_last_query_id()` returns the
SQL cell's query ID automatically.


In [ ]:
SELECT 1 AS test_col, 'hello_from_sql' AS greeting

In [ ]:
sql_qid = nb_last_query_id()
print(f"SQL cell query ID: {sql_qid}")
if sql_qid:
    verify = session.sql(
        f"SELECT * FROM TABLE(RESULT_SCAN('{sql_qid}'))"
    ).to_pandas()
    assert verify.iloc[0]["GREETING"] == "hello_from_sql"
    print(f"[PASS] SQL cell captured + RESULT_SCAN verified: {verify.iloc[0].to_dict()}")
else:
    print("[INFO] SQL cell not captured (non-interactive mode -- no _notify callback)")

## 1b. Multi-Statement SQL Cell

What happens when a SQL cell contains multiple statements?
How many `dataframe_N` variables are created, and which query IDs
does the tracker capture?


In [ ]:
SELECT 'first_stmt' AS which, 100 AS val;
SELECT 'second_stmt' AS which, 200 AS val

In [ ]:
import IPython, json
ns = IPython.get_ipython().user_ns
df_vars = sorted(
    [(k, v) for k, v in ns.items() if k.startswith("dataframe_")],
    key=lambda x: x[0]
)
print(f"All dataframe_N variables ({len(df_vars)}):")
for name, val in df_vars:
    print(f"  {name}: {type(val).__name__} = {val}")

print(f"\nTracker registry after multi-statement SQL cell:")
reg = get_registry()
for i, e in enumerate(reg['all']):
    df = e.get('dataframe') or '(none)'
    print(f"  [{i}] qid={e['query_id'][:30]}  df={df}  sql={e.get('sql', '')[:50]}")

print(f"\nDataframe->QID mappings: {reg['dataframes']}")

diag = {
    "dataframe_vars": [name for name, _ in df_vars],
    "tracker_all": [{"qid": e["query_id"][:40], "df": e.get("dataframe"), "sql": e.get("sql", "")[:80]} for e in reg["all"]],
    "df_mappings": {k: v[:40] for k, v in reg["dataframes"].items()},
}
diag_json = json.dumps(diag, indent=2)
session.sql(f"""
    CREATE OR REPLACE TABLE NOTEBOOK_CI.QUERY_TRACKER_MULTI_STMT_DIAG AS
    SELECT '{len(df_vars)}' AS NUM_DATAFRAME_VARS,
           $$ {diag_json} $$ AS DIAG_JSON,
           CURRENT_TIMESTAMP() AS RUN_AT
""").collect()
print("[INFO] Multi-statement SQL cell diagnostics written to table")

## 2. Python Query Capture + RESULT_SCAN


In [ ]:
session.sql("SELECT 'from_python' AS origin, 42 AS answer").collect()

In [ ]:
py_qid = nb_last_query_id()
print(f"Python query ID: {py_qid}")
assert py_qid is not None, "No query ID captured"

verify = session.sql(
    f"SELECT * FROM TABLE(RESULT_SCAN('{py_qid}'))"
).to_pandas()
assert verify.iloc[0]["ANSWER"] == 42
print("[PASS] Python query capture + Python RESULT_SCAN")

## 3. R Consumes Python Query via RESULT_SCAN

Pass the captured Python query ID to R and retrieve it via
`sfr_query()` with a RESULT_SCAN SQL.


In [ ]:
rscan_sql = f"SELECT * FROM TABLE(RESULT_SCAN('{py_qid}'))"

In [ ]:
%%R -i rscan_sql
library(snowflakeR)
conn <- sfr_connect()
df <- sfr_query(conn, rscan_sql)
cat("R got ANSWER:", df$ANSWER, "\n")
stopifnot(df$ANSWER == 42)
cat("[PASS] R RESULT_SCAN from Python query ID\n")

## 4. R Auto-Lookup via Tracker

Execute a new Python query, then have R call nb_last_query_id()
via reticulate to auto-lookup the most recent query.


In [ ]:
session.sql("SELECT 'cross_lang' AS origin, 99 AS value").collect()

In [ ]:
%%R
qt <- reticulate::import("query_tracker")
auto_qid <- qt$nb_last_query_id()
cat("Auto-lookup query ID:", auto_qid, "\n")
rsql <- sprintf("SELECT * FROM TABLE(RESULT_SCAN('%s'))", auto_qid)
df2 <- sfr_query(conn, rsql)
cat("R auto-lookup VALUE:", df2$VALUE, "\n")
stopifnot(df2$VALUE == 99)
cat("[PASS] R auto-lookup via tracker\n")

## 5. Scala Consumes Python Query via RESULT_SCAN


In [ ]:
session.sql("SELECT 'for_scala' AS origin, 777 AS lucky").collect()

In [ ]:
qid = nb_last_query_id()
print(f"Scala test query ID: {qid}")

In [ ]:
%%scala -i qid
val q = qid.asInstanceOf[String]
val df = sfSession.sql(s"SELECT * FROM TABLE(RESULT_SCAN('$q'))")
df.show()
val rows = df.collect()
assert(rows.length > 0)
println("[PASS] Scala RESULT_SCAN from Python query ID")

## 6. Summary + CI Results


In [ ]:
reg = get_registry()
print(f"Total queries tracked: {len(reg['all'])}")
for i, e in enumerate(reg['all']):
    print(f"  [{i}] {e['query_id'][:30]}  {(e.get('sql') or '')[:50]}")
print("\nAll cross-language tests completed successfully.")

In [ ]:
try:
    _ci = get_active_session()
    _ci.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.QUERY_TRACKER_TEST_RESULTS AS
        SELECT 'query_tracker_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI: PASS")
except Exception:
    pass